# Latent Market Drift — EWMA-PCA (v2, rebuilt)

Three weighting schemes over the same S&P 500 universe, each **$1 invested**:

| Series | Weights | Source |
|---|---|---|
| SPY | free-float market cap | ETF price |
| RSP | equal | ETF price |
| EWMA-PCA | `w = v / sigma`, normalised so `sum(w) = 1` | computed |

`v` is the leading eigenvector of an EWMA covariance matrix of
volatility-standardised returns. Dividing by `sigma` undoes the standardisation
and returns the weights to return space; the sum-to-1 normalisation fixes the
leverage that `||v|| = 1` leaves arbitrary.

**Design rules.**

1. Every quantity used on day `t` is a function of data through `t-1`. The
   weights are fixed *before* today's returns are read, so look-ahead is not
   expressible in the loop. Cell 19 proves this empirically.
2. Point-in-time index membership — no survivorship bias.
3. Simple returns for portfolio aggregation; log returns only for aggregation
   across time.
4. No full-sample constants anywhere. No vol matching.
5. Nothing is silently dropped. Cell 5 names every exclusion.

**Two things this notebook does not do, stated rather than hidden.**

- `S = lambda*S + (1-lambda)*z z'` is an uncentred second moment, `E[zz']`, not
  a covariance. The mean direction is therefore folded into the leading
  eigenvector. That is the confound sitting closest to a drift claim, and it is
  kept deliberately: subtracting a rolling mean would itself be an estimate with
  its own look-ahead surface, and at daily frequency the mean of the
  standardised returns is small beside their dispersion: 0.029 against a
  standard deviation of 1.04, a factor of about 36. Read PC1 as a market *mode*, not as a
  demeaned factor.
- No risk-free rate is subtracted anywhere. Every "drift" here is a total
  return, not an excess return, and the Sharpe-like column in Cell 13 is
  drift/vol with no `r_f` — comparable across the three series, not comparable
  to a published Sharpe ratio.

## Cell 1 — Imports and configuration

In [ ]:
# ═══ CELL 1 ═══
import io
import json
import hashlib
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import requests

warnings.filterwarnings("ignore", category=FutureWarning)

try:
    import yfinance as yf
except ImportError as exc:                      # pragma: no cover
    raise SystemExit("pip install yfinance lxml pyarrow") from exc

try:
    from IPython.display import display
except ImportError:                             # pragma: no cover
    display = print                             # B25: never rely on the builtin


@dataclass(frozen=True)
class Config:
    start_date: str = "2015-01-01"
    end_date:   str = "2026-08-24"     # pinned; do NOT use today()

    vol_half_life: int = 60            # days, EWMA volatility
    pca_half_life: int = 252           # days, EWMA covariance

    burn_in_days: int = 756            # 3 covariance half-lives, discarded
    vol_warmup_obs: int = 20           # per-stock observations before vol is usable
    min_names: int = 30                # fewest active stocks to compute PC1
    max_gap_days: int = 4              # calendar gap above which a return is invalid
    vol_floor: float = 1e-4            # daily; guards halted / illiquid names
    z_clip: float = 10.0               # cap on standardised returns entering S

    # named here so no bare literal appears downstream
    min_net_gross: float = 1e-3        # |sum(w_raw)| floor, as a fraction of gross
    jump_threshold: float = 0.5        # |log return| flagged as a suspect action
    splice_min_cover: float = 0.20     # price coverage inside a membership gap
    splice_min_days: int = 60          # in-window days needed to judge a gap
    boot_n: int = 2000                 # bootstrap replicates
    boot_block: int = 21               # mean block length, stationary bootstrap

    rot_lags: tuple = (21, 63, 252)    # horizons for eigenvector rotation

    trading_days: int = 252
    benchmarks: tuple = ("SPY", "RSP")
    seed: int = 0

    snapshot_dir: str = "./data"

CFG = Config()
Path(CFG.snapshot_dir).mkdir(parents=True, exist_ok=True)
FIGDIR = Path("./figures")
FIGDIR.mkdir(parents=True, exist_ok=True)


def savefig(name):
    """Write each figure to./figures as vector PDF and raster PNG."""
    for ext in ("pdf", "png"):
        plt.savefig(FIGDIR / f"{name}.{ext}", dpi=200, bbox_inches="tight")


LAMBDA_VOL = 0.5 ** (1 / CFG.vol_half_life)
LAMBDA_PCA = 0.5 ** (1 / CFG.pca_half_life)

# Effective sample size of the EWMA covariance: (1+l)/(1-l).
T_EFF = (1 + LAMBDA_PCA) / (1 - LAMBDA_PCA)

print(json.dumps(asdict(CFG), indent=2, default=str))
print(f"\nlambda_vol = {LAMBDA_VOL:.6f}   lambda_pca = {LAMBDA_PCA:.6f}")
print(f"covariance effective sample size ~ {T_EFF:.0f} days")

## Cell 2 — Point-in-time index membership

Wikipedia records dated additions and removals. Starting from today's known
constituents and undoing each change backwards recovers membership on any past
date.

**The two tables now live on different articles.** The "Selected changes"
section was split out of *List of S&P 500 companies* into *Historical components
of the S&P 500*; the first page now contains only the constituents table and a
navbox. Both pages are fetched and every table on them is a candidate.

Tables are located by **content**, never by position — a navbox sits between
them and the index shifts whenever either page is edited. If the expected tables
cannot be found, every table that was seen is printed before raising.

Three details the naive version gets wrong:

- **Undo order.** On 2019-03-19 `FOXA` is recorded as both added and removed
  (21st Century Fox → Fox Corporation, a rename). Undoing all additions before
  reinstating all removals keeps `FOXA` present in the earlier period; the
  reverse order deletes it.
- **Blank cells.** Spin-offs have an addition with no removal; some removals
  have no replacement.
- **Re-entry.** `EQT` left in 2018 and returned in 2022. Chronological replay
  handles this; a static list cannot.

In [ ]:
# ═══ CELL 2 ═══
PAGES = {
    "constituents": "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",
    "changes":      "https://en.wikipedia.org/wiki/Historical_components_of_the_S%26P_500",
}
UA = {"User-Agent": "Mozilla/5.0 (research)"}


def to_yahoo(symbol):
    """Wikipedia writes share classes with a dot; Yahoo uses a dash."""
    return str(symbol).strip().upper().replace(".", "-")


def flatten_columns(cols):
    """Collapse a possibly two-row header into lowercase single strings."""
    if isinstance(cols, pd.MultiIndex):
        out = []
        for c in cols:
            parts = [str(p).strip() for p in c
                     if "Unnamed" not in str(p) and str(p).strip().lower() != "nan"]
            dedup = []
            for p in parts:                       # ('Added','Added') -> 'added'
                if not dedup or dedup[-1].lower() != p.lower():
                    dedup.append(p)
            out.append("_".join(dedup).lower())
        return out
    return [str(c).strip().lower() for c in cols]


WIKI_SNAP = Path(CFG.snapshot_dir) / "wiki_membership.json"


def fetch_pages():
    """Raw HTML of both pages, cached. Membership is an input like prices are:
    left live it drifts with every Wikipedia edit, and the counts reported in
    the paper drift with it. Delete WIKI_SNAP to refresh deliberately."""
    if WIKI_SNAP.exists():
        blob = json.loads(WIKI_SNAP.read_text(encoding="utf-8"))
        print(f"loaded membership snapshot {WIKI_SNAP}")
        print(f"  captured  : {blob['captured']}")
        print(f"  content hash: {blob['hash']}")
        return blob["pages"]

    session = requests.Session()
    pages = {k: session.get(u, headers=UA, timeout=60).text
             for k, u in PAGES.items()}
    digest = hashlib.sha256(
        "".join(pages[k] for k in sorted(pages)).encode("utf-8")).hexdigest()[:16]
    WIKI_SNAP.write_text(json.dumps({
        "captured": pd.Timestamp.utcnow().strftime("%Y-%m-%d %H:%M UTC"),
        "hash": digest, "urls": PAGES, "pages": pages}), encoding="utf-8")
    print(f"wrote membership snapshot {WIKI_SNAP}  hash {digest}")
    return pages


def harvest_tables():
    """Every table on both pages, under both header layouts."""
    found = []
    for page, html in fetch_pages().items():
        for hdr in (0, [0, 1]):
            try:
                tbls = pd.read_html(io.StringIO(html), header=hdr)
            except Exception as exc:
                print(f"  read_html({page}, header={hdr}) failed: {exc}")
                continue
            for i, t in enumerate(tbls):
                found.append((f"{page} header={hdr} idx={i}", t))
    if not found:
        raise RuntimeError("no tables parsed from either Wikipedia page")
    return found


def locate(found):
    """Pick the constituents and changes tables by their column content."""
    current = changes = None
    for label, t in found:
        cols = flatten_columns(t.columns)
        if (current is None and len(t) > 100
                and any(c in ("symbol", "ticker") for c in cols)
                and any("security" in c for c in cols)):
            current = t
        has_add = any(c.startswith("added") for c in cols)
        has_rem = any(c.startswith("removed") for c in cols)
        has_date = any("date" in c for c in cols)
        better = changes is None or (
            any("ticker" in c for c in cols)
            and not any("ticker" in c for c in flatten_columns(changes.columns)))
        if has_add and has_rem and has_date and len(t) > 50 and better:
            changes = t

    if current is None or changes is None:
        print("Could not locate the expected tables. What the pages contained:\n")
        for label, t in found:
            print(f"  {label:34s} rows={len(t):4d} cols={flatten_columns(t.columns)[:6]}")
        missing = ("constituents " if current is None else "") + \
                  ("changes " if changes is None else "")
        raise RuntimeError(f"missing {missing}table (see listing above)")
    return current, changes


def parse_current(tbl):
    df = tbl.copy()
    df.columns = flatten_columns(df.columns)
    sym = next(c for c in df.columns if c in ("symbol", "ticker"))
    name = next((c for c in df.columns if "security" in c), sym)
    out = pd.DataFrame({"ticker": df[sym].map(to_yahoo), "security": df[name]})
    return out.dropna(subset=["ticker"]).drop_duplicates("ticker").reset_index(drop=True)


def parse_changes(tbl):
    """Normalise to date / added / removed plus both security names."""
    df = tbl.copy()
    df.columns = flatten_columns(df.columns)
    cols = list(df.columns)

    c_date = next((c for c in cols if "date" in c), cols[0])
    add_cols = [c for c in cols if c.startswith("added")]
    rem_cols = [c for c in cols if c.startswith("removed")]
    if not add_cols or not rem_cols:
        raise ValueError(f"unexpected changes-table columns: {cols}")

    c_add = next((c for c in add_cols if "ticker" in c), add_cols[0])
    c_rem = next((c for c in rem_cols if "ticker" in c), rem_cols[0])
    c_add_s = next((c for c in add_cols if "security" in c), None)
    c_rem_s = next((c for c in rem_cols if "security" in c), None)

    dates = pd.to_datetime(df[c_date], format="mixed", errors="coerce")
    if dates.notna().mean() < 0.5:                    # tolerate older pandas
        dates = pd.to_datetime(df[c_date], errors="coerce")

    def clean(s):
        s = s.astype("string").str.strip()
        return s.where(s.notna() & (s != "")
                       & (~s.str.lower().isin(["nan", "ticker", "none", "—", "-"])))

    out = pd.DataFrame({
        "date": dates,
        "added": clean(df[c_add]).map(lambda x: to_yahoo(x) if pd.notna(x) else pd.NA),
        "removed": clean(df[c_rem]).map(lambda x: to_yahoo(x) if pd.notna(x) else pd.NA),
        "added_security": clean(df[c_add_s]) if c_add_s else pd.NA,
        "removed_security": clean(df[c_rem_s]) if c_rem_s else pd.NA,
    })
    out = out.dropna(subset=["date"]).reset_index(drop=True)
    if out.empty:
        raise ValueError(f"changes table parsed to zero dated rows; columns were {cols}")
    return out


found = harvest_tables()
print(f"tables harvested: {len(found)}")
for label, t in found:
    print(f"   {label:34s} rows={len(t):4d}")

current_tbl, changes_tbl = locate(found)
current_df = parse_current(current_tbl)
changes_df = parse_changes(changes_tbl)

print(f"\ncurrent constituents : {len(current_df)}")
print(f"recorded changes     : {len(changes_df)}  "
      f"({changes_df.date.min():%Y-%m-%d} to {changes_df.date.max():%Y-%m-%d})")
print(f"additions / removals : {changes_df.added.notna().sum()} / "
      f"{changes_df.removed.notna().sum()}")
display(changes_df.head(8))

## Cell 3 — Membership timeline

The timeline is replayed backwards, then expanded into a (dates × tickers)
boolean matrix. The universe downloaded in Cell 4 is the **union of everything
that was a member at any point in the window**, not today's list — that is what
removes the survivorship half of B1.

In [ ]:
# ═══ CELL 3 ═══
def build_membership_timeline(current_members, changes, as_of):
    """Undo dated changes backwards from a known current snapshot."""
    changes = changes.dropna(subset=["date"]).sort_values("date")
    cursor = set(current_members)
    timeline = []

    groups = sorted(changes.groupby("date"), key=lambda kv: kv[0], reverse=True)
    for date, block in groups:
        # A change dated after as_of is still UNDONE, never skipped. The cursor
        # starts from the current constituent list, which already reflects it,
        # so skipping would carry a later addition back across the whole sample
        # and drop a later removal from it. Only changes at or before as_of
        # produce a timeline entry.
        if date <= as_of:
            timeline.append((date, frozenset(cursor)))

        prior = set(cursor)
        for tk in block["added"].dropna():        # undo additions FIRST
            prior.discard(tk)
        for tk in block["removed"].dropna():      # then reinstate removals
            prior.add(tk)
        cursor = prior

    timeline.append((pd.Timestamp("1900-01-01"), frozenset(cursor)))
    timeline.reverse()
    return timeline


def membership_matrix(timeline, dates, tickers):
    tickers = list(tickers)
    col_of = {t: i for i, t in enumerate(tickers)}
    out = np.zeros((len(dates), len(tickers)), dtype=bool)
    starts = np.array([d.value for d, _ in timeline])

    for row, dt in enumerate(dates):
        i = max(np.searchsorted(starts, dt.value, side="right") - 1, 0)
        for tk in timeline[i][1]:
            j = col_of.get(tk)
            if j is not None:
                out[row, j] = True
    return pd.DataFrame(out, index=dates, columns=tickers)


def security_names(changes):
    """{ticker: [name,...]} over every add/remove record. Reporting only."""
    names = {}
    for side in ("added", "removed"):
        sec = f"{side}_security"
        if sec not in changes.columns:
            continue
        for tk, nm in zip(changes[side], changes[sec]):
            if pd.isna(tk) or pd.isna(nm):
                continue
            names.setdefault(str(tk), set()).add(str(nm).strip())
    return {tk: sorted(v) for tk, v in names.items()}


def membership_gaps(changes):
    """{ticker: [(removed_on, re_added_on),...]} -- left the index, came back."""
    ev = {}
    for _, row in changes.dropna(subset=["date"]).sort_values("date").iterrows():
        if pd.notna(row.get("added")):
            ev.setdefault(str(row["added"]), []).append((row["date"], "add"))
        if pd.notna(row.get("removed")):
            ev.setdefault(str(row["removed"]), []).append((row["date"], "rem"))

    out = {}
    for tk, events in ev.items():
        gaps, open_rem = [], None
        for date, kind in sorted(events, key=lambda e: (e[0], e[1])):
            if kind == "rem":
                open_rem = date
            elif open_rem is not None and date > open_rem:
                gaps.append((open_rem, date))
                open_rem = None
        if gaps:
            out[tk] = gaps
    return out


AS_OF = pd.Timestamp(CFG.end_date)
TIMELINE = build_membership_timeline(current_df.ticker, changes_df, AS_OF)
NAMES = security_names(changes_df)
GAPS = membership_gaps(changes_df)

win_start, win_end = pd.Timestamp(CFG.start_date), AS_OF
_universe = set()
for i, (seg_start, members) in enumerate(TIMELINE):
    seg_end = TIMELINE[i + 1][0] if i + 1 < len(TIMELINE) else win_end
    if seg_end >= win_start and seg_start <= win_end:   # segment overlaps window
        _universe |= members
UNIVERSE = sorted(_universe)

print(f"timeline segments      : {len(TIMELINE)}")
print(f"tickers ever a member  : {len(UNIVERSE)}")
print(f"members today          : {len(TIMELINE[-1][1])}")
print(f"symbols with a gap     : {len(GAPS)}")

_probe = pd.date_range(CFG.start_date, CFG.end_date, freq="12ME")
_counts = membership_matrix(TIMELINE, _probe, UNIVERSE).sum(axis=1)
print("\nmembership count over time (should hover near 500):")
print(_counts.to_string())

## Cell 4 — Download and snapshot

Prices are fetched **once** and written to disk with a content hash. Everything
downstream reads the snapshot, so results do not change between runs and
download flakiness cannot silently alter the universe. Delete the file to
re-fetch.

`end_date` is pinned in Cell 1 rather than taken from `today()`, for the same
reason.

In [ ]:
# ═══ CELL 4 ═══
SNAP = Path(CFG.snapshot_dir) / f"prices_{CFG.start_date}_{CFG.end_date}.parquet"
FAIL = Path(CFG.snapshot_dir) / f"failed_{CFG.start_date}_{CFG.end_date}.json"


def download_prices(tickers, start, end, chunk=100, retries=3):
    """Adjusted closes, plus the list that failed. Chunked, with back-off.

    Never shrink the universe to dodge rate-limiting -- retry it.
    """
    import time
    frames, failed = [], []
    tickers = list(tickers)

    for i in range(0, len(tickers), chunk):
        batch = tickers[i:i + chunk]
        print(f"  {i + 1:4d}-{i + len(batch):4d} of {len(tickers)}", flush=True)
        raw = None
        for attempt in range(retries):
            try:
                raw = yf.download(batch, start=start, end=end, auto_adjust=True,
                                  progress=False, threads=True)
                if raw is not None and not raw.empty:
                    break
            except Exception as exc:
                print(f"     attempt {attempt + 1} failed: {exc}")
            time.sleep(2 ** attempt)
        if raw is None or raw.empty:
            failed.extend(batch)
            continue

        close = raw["Close"] if isinstance(raw.columns, pd.MultiIndex) else raw[["Close"]]
        if not isinstance(raw.columns, pd.MultiIndex):
            close.columns = batch[:1]

        got = [c for c in close.columns if close[c].notna().any()]
        failed.extend(sorted(set(batch) - set(got)))
        if got:
            frames.append(close[got])

    prices = pd.concat(frames, axis=1).sort_index()
    return prices.loc[~prices.index.duplicated()], failed


if SNAP.exists():
    prices_raw = pd.read_parquet(SNAP)
    failed_tickers = json.loads(FAIL.read_text()) if FAIL.exists() else []
    print(f"loaded snapshot {SNAP}")
else:
    print(f"downloading {len(UNIVERSE)} constituents + {len(CFG.benchmarks)} benchmarks...")
    prices_raw, failed_tickers = download_prices(
        UNIVERSE + list(CFG.benchmarks), CFG.start_date, CFG.end_date)
    prices_raw.to_parquet(SNAP)
    FAIL.write_text(json.dumps(failed_tickers))
    print(f"wrote snapshot {SNAP}")

prices_raw.index = pd.to_datetime(prices_raw.index)
if getattr(prices_raw.index, "tz", None) is not None:      # naive index raises
    prices_raw.index = prices_raw.index.tz_localize(None)

digest = hashlib.sha256(
    pd.util.hash_pandas_object(prices_raw.fillna(0.0), index=True).values.tobytes()
).hexdigest()[:16]

print(f"\nshape          : {prices_raw.shape}")
print(f"date range     : {prices_raw.index.min():%Y-%m-%d} to {prices_raw.index.max():%Y-%m-%d}")
print(f"failed tickers : {len(failed_tickers)}")
print(f"content hash   : {digest}")

## Cell 5 — Integrity report

Every exclusion is named with a reason.

- **Failed downloads** were genuine index members with no usable history, mostly
  delisted acquisitions. Their absence biases the universe toward survivors, so
  the count is reported rather than buried.
- **Spliced listings** are symbols whose price history joins two *different*
  companies. The discriminator is the data, not the name: comparing security
  names cannot separate `Bemis` / `Bemis Company` (one company, two spellings)
  from `SanDisk` / `Sandisk` (two companies, differing only in case). What does
  separate them is whether the symbol kept trading through its index gap. `EQT`
  left the index in 2018 and returned in 2022 but traded every day in between,
  so it is kept, with the gap reflected in its membership. `CEG`, `DOW`, `Q` and
  `SNDK` have almost no prices inside their gaps and are dropped.
  A splice that predates the study window is harmless — `DELL`'s old listing
  ended in 2013, so inside a 2015-2026 sample every `DELL` price is one listing.
- **Big jumps** flag suspected unadjusted corporate actions for inspection; they
  are reported, not dropped.

In [ ]:
# ═══ CELL 5 ═══
def find_spliced_listings(gaps, prices, min_cover, min_days):
    """Symbols whose price history splices two different listings."""
    out = {}
    for tk, spans in gaps.items():
        if tk not in prices.columns:
            continue
        s = prices[tk]
        if not s.notna().any():
            continue                                 # failed download, not a splice
        for lo, hi in spans:
            window = s.loc[(s.index > lo) & (s.index < hi)]
            if len(window) < min_days:
                continue                             # gap mostly outside the sample
            cover = float(window.notna().mean())
            if cover < min_cover:
                out[tk] = (lo, hi, cover)
                break
    return out


bench_cols = [b for b in CFG.benchmarks if b in prices_raw.columns]
missing_bench = [b for b in CFG.benchmarks if b not in prices_raw.columns]
if missing_bench:
    raise RuntimeError(f"benchmark download failed: {missing_bench}")

# Restricted to the universe, not simply to everything in the snapshot.
# The snapshot is additive across runs, so a column downloaded for an
# earlier universe can survive in it; counting those would overstate how
# many index members have price history and would carry dead columns
# into the panel.
stock_cols = [c for c in prices_raw.columns
              if c not in CFG.benchmarks and c in set(UNIVERSE)]

# Derive the failures from the snapshot itself rather than from the cached
# JSON. The two can disagree: the snapshot may have been written by a later
# run, leaving the JSON describing a download that no longer corresponds to
# the data in use, and the count is reported in the paper.
failed_tickers = sorted(set(UNIVERSE) - set(prices_raw.columns))

spliced = find_spliced_listings(GAPS, prices_raw[stock_cols],
                                CFG.splice_min_cover, CFG.splice_min_days)

lr = np.log(prices_raw[stock_cols].where(prices_raw[stock_cols] > 0)).diff()
jumps = (lr.abs() > CFG.jump_threshold).sum()
jumpy = sorted(jumps[jumps > 0].index)

report = pd.DataFrame({
    "n_obs":       prices_raw[stock_cols].notna().sum(),
    "first":       prices_raw[stock_cols].apply(lambda s: s.first_valid_index()),
    "last":        prices_raw[stock_cols].apply(lambda s: s.last_valid_index()),
    "pct_missing": (prices_raw[stock_cols].isna().mean() * 100).round(1),
    "big_jumps":   jumps,
    "spliced":     [c in spliced for c in stock_cols],
})

print(f"members ever in window : {len(UNIVERSE)}")
print(f"downloaded             : {len(stock_cols)}")
print(f"failed download        : {len(failed_tickers)}")
if failed_tickers:
    print("   " + ", ".join(sorted(failed_tickers)))

print(f"\nspliced listings (dropped): {len(spliced)}")
for tk, (lo, hi, cov) in sorted(spliced.items()):
    print(f"   {tk:6s} gap {lo:%Y-%m-%d} -> {hi:%Y-%m-%d}  coverage {cov:.1%}   {NAMES.get(tk)}")

kept_gaps = {tk: g for tk, g in GAPS.items() if tk in stock_cols and tk not in spliced}
print(f"\nre-entrants kept (index gap, continuous trading): {len(kept_gaps)}")
print("   " + ", ".join(sorted(kept_gaps)[:25]))
# with the coverage that cleared each of them, since the paper quotes the
# lowest of these as evidence that the test separates splices from genuine
# re-entrants rather than dropping on the gap alone
print("   coverage inside the index gap, on the splice test's own window:")
for tk in sorted(kept_gaps):
    _s = prices_raw[tk]
    for lo, hi in kept_gaps[tk]:
        _w = _s.loc[(_s.index > lo) & (_s.index < hi)]
        print(f"      {tk:6s} gap {lo:%Y-%m-%d} -> {hi:%Y-%m-%d}  "
              f"{len(_w):5d} in-window days  "
              + ("too short to judge" if len(_w) < CFG.splice_min_days
                 else f"{_w.notna().mean():.1%}"))

print(f"\n|log return| > {CFG.jump_threshold} (check for unadjusted actions): {len(jumpy)}")
if jumpy:
    print("   " + ", ".join(jumpy[:25]) + ("..." if len(jumpy) > 25 else ""))

KEEP = [c for c in stock_cols if c not in spliced]
print(f"\nretained for analysis   : {len(KEEP)}")
display(report.sort_values("pct_missing", ascending=False).head(12))

## Cell 6 — Returns

Simple returns are the primitive, because `sum(w * r)` is only a portfolio
return in simple space. Log returns are derived for aggregation across time.

A return is marked **invalid** — not zero, not forward-filled — when either
endpoint price is missing or non-positive, or when the calendar gap between them
exceeds `max_gap_days`. Without that last check a multi-day price gap becomes a
single enormous "daily" return and manufactures fat tails.

Note what is *not* here: `returns.mean(axis=1)` as an equal-weighted index. The
mean of log returns is not the log of a portfolio return, and at this panel's
dimensions the difference was 3.63 pp/yr. RSP replaces it.

In [ ]:
# ═══ CELL 6 ═══
def simple_returns(prices, max_gap_days):
    prices = prices.astype(float)
    prev = prices.shift(1)
    rets = prices / prev - 1.0

    gap = prices.index.to_series().diff().dt.days.to_numpy()
    gap_ok = (gap <= max_gap_days)
    gap_ok[0] = False

    valid = (prices.gt(0) & prev.gt(0) & prices.notna() & prev.notna()
             & pd.DataFrame(np.repeat(gap_ok.reshape(-1, 1), prices.shape[1], axis=1),
                            index=prices.index, columns=prices.columns))
    return rets.where(valid), valid


px = prices_raw[KEEP]
rets, valid = simple_returns(px, CFG.max_gap_days)

bench = prices_raw[bench_cols]
bench_rets, bench_valid = simple_returns(bench, CFG.max_gap_days)

DATES = rets.index
MEMBERS = membership_matrix(TIMELINE, DATES, KEEP)

print(f"return panel   : {rets.shape}")
print(f"valid cells    : {valid.to_numpy().sum():,} of {valid.size:,} "
      f"({100 * valid.to_numpy().mean():.1f}%)")
print(f"invalidated by gap/price rules: "
      f"{int((rets.isna() & px.notna()).to_numpy().sum()):,}")
print(f"\nactive members per day: min {MEMBERS.sum(axis=1).min()}, "
      f"median {int(MEMBERS.sum(axis=1).median())}, max {MEMBERS.sum(axis=1).max()}")

## Cell 7 — EWMA volatility (lagged)

Two properties matter.

**Lagging.** `sigma` is recorded *before* today's update, so `sigma_{t-1}`
standardises `r_t`. Standardising with a contemporaneous `sigma_t` that already
contains `r_t^2` imposes a hard cap `|z| <= 1/sqrt(1-lambda) = 9.33`, censoring
exactly the crisis tails the study is about. Cell 19 checks both halves of
this claim.

**Missing data freezes rather than poisons.** The update applies only where the
observation is valid, so one missing return no longer turns a stock's variance
into `NaN` for the rest of the sample.

There is also no 60-day seeded initial variance and no `ddof` mismatch
: the recursion is seeded by each stock's own first valid observation.

In [ ]:
# ═══ CELL 7 ═══
def ewma_vol_lagged(returns, valid, half_life, floor, warmup):
    lam = 0.5 ** (1.0 / half_life)
    r = returns.to_numpy(dtype=float)
    m = valid.to_numpy(dtype=bool)
    n_t, n_k = r.shape

    var = np.full(n_k, np.nan)
    out = np.full((n_t, n_k), np.nan)
    cnt = np.zeros(n_k, dtype=int)
    ready = np.zeros((n_t, n_k), dtype=bool)

    for t in range(n_t):
        out[t] = var                               # state BEFORE today
        ready[t] = cnt >= warmup

        obs = m[t]
        x = np.where(obs, r[t], 0.0) ** 2
        fresh = obs & ~np.isfinite(var)            # first observation
        var = np.where(fresh, x, var)
        upd = obs & ~fresh
        var = np.where(upd, lam * var + (1.0 - lam) * x, var)
        cnt += obs

    vol = np.sqrt(out)
    vol = np.where(np.isfinite(vol), np.maximum(vol, floor), np.nan)   # B21
    return pd.DataFrame(np.where(ready, vol, np.nan),
                        index=returns.index, columns=returns.columns)


vol = ewma_vol_lagged(rets, valid, CFG.vol_half_life,
                      CFG.vol_floor, CFG.vol_warmup_obs)

z_all = (rets / vol).to_numpy()
z_all = z_all[np.isfinite(z_all)]
cap = 1 / np.sqrt(1 - LAMBDA_VOL)

print(f"annualised vol, median stock : {np.nanmedian(vol.to_numpy()) * np.sqrt(252):.1%}")
print(f"max |z|                      : {np.abs(z_all).max():.2f}")
print(f"cap that contemporaneous standardisation would impose: {cap:.2f}")
print(f"observations beyond that cap : {(np.abs(z_all) > cap).sum():,} "
      f"({100 * (np.abs(z_all) > cap).mean():.3f}%)  a contemporaneous sigma would censor these")

## Cell 8 — Streaming EWMA PCA

The core loop. Per day, in this exact order:

1. eigendecompose `S_{t-1}` over the active names — no knowledge of today
2. sign-align against yesterday's eigenvector (max inner product), which a
   "sum of loadings" rule gets wrong whenever the loadings are mixed in sign
3. `w = v / sigma_{t-1}`, then `w /= w.sum()` — unit net exposure (D1)
4. realise `r_pc1 = w. r_t` — **the only place today's returns are read**
5. update `S` with today's standardised returns

Because the weights are fixed at step 3, before step 4 ever touches `r_t`,
look-ahead is not expressible in this loop. The ordering is what makes that
true: updating `S` with `z_t` before decomposing it would read day `t` to build
the direction that then prices day `t`.

**Re-seeding entrants.** `S` decays globally by `lambda` every day, including
for names that are not yet members and are contributing `z = 0`. A name joining
after eight years would arrive with a diagonal of `lambda^2000 ~ 0.004` instead
of the intended 1.0. Since `z` is volatility-standardised every diagonal should
sit near 1, so a decayed diagonal is an artefact of the recursion rather than a
measurement. This is a *consistency* fix, not a bias fix, and it is easy to get
backwards: an entrant's off-diagonal entries are near zero either way, so
re-seeding only changes the variance they are divided by, and raising it makes
the implied correlations smaller. On synthetic data fifty days after entry they
read 0.046 re-seeded against 0.107 not re-seeded, with a true value of 0.325 —
both far below, and re-seeding moves away from it. What it buys is that an
entrant carries the prior every incumbent carried on day 0, rather than one
scaled by how long the sample ran before it arrived. So a name's row and
column are reset to the identity prior when it
**enters the index** — a membership edge, never a one-day validity gap, which
would otherwise wipe a stock's history for one bad print. Setting row/col `i` to
`e_i` keeps `S` positive semi-definite.

This also repairs the burn-in argument in Cell 9: discarding 756 days is
justified by the identity prior decaying to `lambda^756`, but that holds only
for names present on day 0. Without re-seeding the same defect recurs at every
later entry date, where no amount of burn-in at the start can reach it.

In [ ]:
# ═══ CELL 8 ═══
def principal_angle_deg(a, b):
    """Angle in degrees between two directions, ignoring sign.

    Computed as 2*atan2(|u-w|, |u+w|) on unit vectors rather than arccos(u.w):
    the two agree mathematically, but arccos is ill-conditioned at small angles
    and these angles are small by construction.
    """
    a = np.asarray(a, dtype=float); b = np.asarray(b, dtype=float)
    na, nb = float(np.linalg.norm(a)), float(np.linalg.norm(b))
    if na <= 0 or nb <= 0:
        return np.nan
    u, w = a / na, b / nb
    if float(np.dot(u, w)) < 0:          # compare the lines, not the vectors
        w = -w
    return float(np.degrees(2.0 * np.arctan2(
        float(np.linalg.norm(u - w)), float(np.linalg.norm(u + w)))))


def spectral_separation(evals, k, t_eff):
    """Where the spectrum sits relative to the Marčenko-Pastur noise bulk.

    Eigenvalues are rescaled to sum to k, as a correlation matrix's would, so
    the comparison is on the right scale even though S is an uncentred second
    moment whose diagonal is only approximately one.
    """
    evals = np.asarray(evals, dtype=float)
    tot = float(evals.sum())
    if tot <= 0 or k < 2 or not np.isfinite(tot):
        return np.nan, np.nan, np.nan, np.nan
    ev = evals * (k / tot)
    q = k / t_eff
    edge = (1.0 + np.sqrt(q)) ** 2
    above = ev > edge
    return float(ev.max()), float(edge), int(above.sum()), float(ev[above].sum() / k)


def run_streaming_pca(returns, valid, vol, members, half_life,
                      min_names, z_clip, min_net_gross, rot_lags, reseed=True):
    lam = 0.5 ** (1.0 / half_life)
    dates = returns.index
    tickers = list(returns.columns)
    n_k, n = len(tickers), len(dates)

    R = returns.to_numpy(dtype=float)
    V = valid.to_numpy(dtype=bool)
    SG = vol.to_numpy(dtype=float)
    M = members.to_numpy(dtype=bool)

    S = np.eye(n_k)
    v_prev = np.zeros(n_k)
    w_prev = np.zeros(n_k)
    prev_member = np.zeros(n_k, dtype=bool)

    factor      = np.full(n, np.nan)
    pct_pos     = np.full(n, np.nan)
    var_exp     = np.full(n, np.nan)
    turnover    = np.full(n, np.nan)
    short_gross = np.full(n, np.nan)
    top10       = np.full(n, np.nan)
    n_act       = np.zeros(n, dtype=int)

    # --- loading trajectory and spectrum ---
    lam1_n    = np.full(n, np.nan)
    mp_edge   = np.full(n, np.nan)
    n_above   = np.full(n, np.nan)
    shr_above = np.full(n, np.nan)
    rot = {h: np.full(n, np.nan) for h in rot_lags}

    t_eff = (1.0 + lam) / (1.0 - lam)
    L = max(rot_lags) + 1              # circular buffer of past eigenvectors
    buf_v = np.zeros((L, n_k))
    buf_a = np.zeros((L, n_k), dtype=bool)
    # Day stamp per slot. Slots are written only on usable days, so without the
    # stamp a wrapped slot could serve the eigenvector from t-h-L instead of t-h.
    buf_t = np.full(L, -1, dtype=np.int64)

    last = None                     # (S_used, idx, v) for the final usable day

    for t in range(n):
        # --- re-seed entrants to the identity prior (membership edge only) ---
        entering = M[t] & ~prev_member
        if reseed and entering.any():
            e = np.flatnonzero(entering)
            S[e, :] = 0.0
            S[:, e] = 0.0
            S[e, e] = 1.0
        prev_member = M[t].copy()

        active = M[t] & V[t] & np.isfinite(SG[t])
        k = int(active.sum())
        n_act[t] = k

        if k >= min_names:
            idx = np.flatnonzero(active)
            S_used = S[np.ix_(idx, idx)]

            # 1. eigenvector of YESTERDAY's covariance
            evals, evecs = np.linalg.eigh(S_used)
            v = evecs[:, -1]

            # 2. continuity of sign across days
            ref = v_prev[idx]
            if np.any(ref):
                if np.dot(v, ref) < 0:
                    v = -v
            elif v.sum() < 0:                      # first usable day only
                v = -v

            v_full = np.zeros(n_k); v_full[idx] = v
            v_prev = v_full

            var_exp[t] = evals[-1] / evals.sum()
            lam1_n[t], mp_edge[t], n_above[t], shr_above[t] = spectral_separation(
                evals, k, t_eff)

            # rotation against past eigenvectors, on the common active set.
            # Only past vectors are read, so this stays causal.
            for h in rot_lags:
                if t - h < 0:
                    continue
                src = (t - h) % L
                if buf_t[src] != t - h:        # that day produced no PC1
                    continue
                common = active & buf_a[src]
                if common.sum() >= min_names:
                    rot[h][t] = principal_angle_deg(v_full[common], buf_v[src][common])
            slot = t % L
            buf_v[slot] = v_full; buf_a[slot] = active; buf_t[slot] = t

            # 3. weights: back to return space, then unit NET exposure
            raw = v / SG[t, idx]
            denom = raw.sum()
            gross = np.abs(raw).sum()
            if np.isfinite(denom) and abs(denom) > min_net_gross * gross:
                w = raw / denom
                w_full = np.zeros(n_k); w_full[idx] = w

                # reported on the WEIGHTS, not on v: the eigenvector's sign is
                # arbitrary and w is invariant to it (see the note above)
                pct_pos[t]     = float(np.mean(w > 0) * 100.0)
                turnover[t]    = np.abs(w_full - w_prev).sum()
                short_gross[t] = float(np.abs(w[w < 0]).sum())
                top10[t]       = float(np.sort(w)[::-1][:10].sum())

                # 4. realise today's return
                factor[t] = float(np.dot(w, R[t, idx]))
                w_prev = w_full
                last = (S_used.copy(), idx.copy(), v.copy(), SG[t, idx].copy())

        # 5. advance the covariance state
        z = np.zeros(n_k)
        if active.any():
            z[active] = np.clip(R[t, active] / SG[t, active], -z_clip, z_clip)
        S = lam * S + (1.0 - lam) * np.outer(z, z)

    cols = {
        "pc1":            factor,
        "pct_positive":   pct_pos,
        "var_explained":  var_exp,
        "n_active":       n_act,
        "turnover":       turnover,
        "short_gross":    short_gross,
        "top10_weight":   top10,
        "lam1_norm":      lam1_n,
        "mp_edge":        mp_edge,
        "n_above_mp":     n_above,
        "share_above_mp": shr_above,
    }
    for h in rot_lags:
        cols[f"rot_{h}d"] = rot[h]
    return pd.DataFrame(cols, index=dates), S, last


print("running streaming PCA...")
diag, S_final, LAST = run_streaming_pca(
    rets, valid, vol, MEMBERS, CFG.pca_half_life,
    CFG.min_names, CFG.z_clip, CFG.min_net_gross, CFG.rot_lags)
print("done.")
print(f"days with a usable PC1: {diag['pc1'].notna().sum():,} of {len(diag):,}")

# Persist the diagnostics so every figure and every number reported in the
# paper can be traced to the run that produced it without recomputing the loop.
diag.to_parquet(Path(CFG.snapshot_dir) / "diag.parquet")
print("wrote data/diag.parquet")

## Cell 9 — Assemble and discard burn-in

The covariance starts at the identity. At a 252-day half-life that prior still
carries 50% weight after one year, so early "variance explained" is an artefact
of initialisation rather than a fact about markets. Three half-lives
(756 days) leaves the prior under 13% and is discarded.

In [ ]:
# ═══ CELL 9 ═══
panel = pd.DataFrame({
    "EWMA_PCA": diag["pc1"],
    "SPY":      bench_rets["SPY"],
    "RSP":      bench_rets["RSP"],
}).dropna()

full_start = panel.index.min()
cut = DATES[min(CFG.burn_in_days, len(DATES) - 1)]
BURN = max(cut, full_start)

ret_df = panel.loc[panel.index >= BURN].copy()
log_df = np.log1p(ret_df)
diag_post = diag.loc[diag.index >= BURN]

prior_weight = LAMBDA_PCA ** CFG.burn_in_days
print(f"burn-in discarded through  : {BURN:%Y-%m-%d} ({CFG.burn_in_days} days)")
print(f"identity prior weight left : {prior_weight:.3%}")
print(f"analysis window            : {ret_df.index.min():%Y-%m-%d} to "
      f"{ret_df.index.max():%Y-%m-%d}  ({len(ret_df):,} days)")
print("\nannualised volatility (all three are $1-invested, no rescaling):")
print((ret_df.std() * np.sqrt(CFG.trading_days)).map("{:.2%}".format).to_string())

## Cell 10 — Is PC1 the market mode?

Correlation is scale-invariant, so this diagnostic was valid even in the old
notebook. It is the check that licenses reading PC1 as "the market".

In [ ]:
# ═══ CELL 10 ═══
print("full-sample correlations")
print(ret_df.corr().round(4).to_string())

roll = pd.DataFrame({
    "PC1 vs SPY": ret_df["EWMA_PCA"].rolling(CFG.trading_days).corr(ret_df["SPY"]),
    "PC1 vs RSP": ret_df["EWMA_PCA"].rolling(CFG.trading_days).corr(ret_df["RSP"]),
    "SPY vs RSP": ret_df["SPY"].rolling(CFG.trading_days).corr(ret_df["RSP"]),
}).dropna()

print("")
print("rolling 252-day correlations, distribution over the window")
display(roll.quantile([0.25, 0.5, 0.75]).round(4))
print("minima: " + ", ".join(f"{c} {roll[c].min():.3f} on {roll[c].idxmin():%Y-%m-%d}"
                             for c in roll.columns))

fig, ax = plt.subplots(figsize=(13, 4.5))
for c in roll.columns:
    ax.plot(roll.index, roll[c], lw=1.0, label=c)
ax.set_ylim(0.5, 1.02)
ax.axhline(0.9, ls=":", c="gray", alpha=0.6)
ax.set_title("Rolling 252-day correlation")
ax.set_ylabel("correlation"); ax.legend(loc="lower left"); ax.grid(alpha=0.25)
plt.tight_layout(); savefig("fig_rolling_correlation")
plt.show()

print("\nminimum rolling correlation with SPY: "
      f"{roll['PC1 vs SPY'].min():.3f}  on {roll['PC1 vs SPY'].idxmin():%Y-%m-%d}")

## Cell 11 — PC1 diagnostics through time

**"Percent positive" is measured on the weights, not on the eigenvector.**
`eigh` returns an arbitrary sign — `v` and `-v` are the same mode — and the
normalisation `w = (v/sigma) / sum(v/sigma)` is invariant to it, because
flipping `v` flips the denominator too. Reporting the sign of `v` therefore
reports a coin flip: on this data the raw eigenvector comes out about 99%
*negative*, the net exposure `sum(v/sigma)` is negative with it, and the
resulting weights are about 99% *positive* with `sum(w) = +1`. The portfolio and
every return are identical either way; only the diagnostic differs. The share of
positive **weights** is the quantity with a meaning — it says whether this is a
market mode that holds nearly everything long — so that is what is plotted and
printed.

In [ ]:
# ═══ CELL 11 ═══
fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)

axes[0].plot(diag_post.index, diag_post["pct_positive"], lw=0.7, color="steelblue")
axes[0].axhline(50, ls="--", c="gray", alpha=0.5)
axes[0].set_ylim(0, 105); axes[0].set_ylabel("% positive")
axes[0].set_title("PC1 — share of portfolio weights with positive sign")

axes[1].plot(diag_post.index, diag_post["var_explained"] * 100, lw=0.7, color="darkorange")
axes[1].set_ylabel("variance explained (%)")
axes[1].set_title("PC1 — variance explained (slightly biased, see cell 21)")

axes[2].plot(diag_post.index, diag_post["n_active"], lw=0.7, color="seagreen")
axes[2].set_ylabel("stocks"); axes[2].set_xlabel("date")
axes[2].set_title("Active universe size")

for a in axes: a.grid(alpha=0.25)
plt.tight_layout(); savefig("fig_pc1_diagnostics")
plt.show()

pp = diag_post["pct_positive"]
print(f"positive weights, median   : {pp.median():.2f}%")
print(f"positive weights, min/max  : {pp.min():.2f}% / {pp.max():.2f}%")
print(f"days below 90% positive    : {int((pp < 90).sum())} of {int(pp.notna().sum())}")

n_bar = diag_post["n_active"].median()
print("\nvariance explained: median %.2f%%   range %.2f%% to %.2f%%"
      % (100 * diag_post["var_explained"].median(),
         100 * diag_post["var_explained"].min(),
         100 * diag_post["var_explained"].max()))
print(f"\nmedian active names        : {n_bar:.0f}")
print(f"covariance effective sample: {T_EFF:.0f} days")
print(f"N / T_eff                  : {n_bar / T_EFF:.2f}")
print("\nA ratio much above ~0.1 puts the eigenvalue spectrum in Marčenko-Pastur")
print("territory: PC1 itself is a strong enough spike to survive, and the")
print("sampling bias in its reported variance share is about q/(lambda-1), which")
print("at this spike size is roughly half a per cent in relative terms.")

## Cell 12 — Weight diagnostics

PC1 is only a credible benchmark if its weights are implementable. The
eigenvector moves daily, so turnover is the number that decides whether this
reads as an index or as a high-frequency strategy. Negative loadings are kept
rather than clipped, because clipping would misrepresent the eigenvector — but
the short exposure they imply is reported.

`sum(w) = 1` pins **net** exposure, not gross. A series can be net $1 and still
require a large long book against a large short book, and that gross figure is
what decides whether this could replace SPY as a hedging instrument.

In [ ]:
# ═══ CELL 12 ═══
fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)

axes[0].plot(diag_post.index, diag_post["turnover"], lw=0.6, color="indianred")
axes[0].set_ylabel("daily turnover"); axes[0].set_yscale("log")
axes[0].set_title("PC1 portfolio — daily turnover, purchases and sales (sum |dw|)")

axes[1].plot(diag_post.index, diag_post["short_gross"] * 100, lw=0.6, color="purple")
axes[1].set_ylabel("gross short (%)")
axes[1].set_title("Short exposure implied by negative weights")

axes[2].plot(diag_post.index, diag_post["top10_weight"] * 100, lw=0.6, color="teal")
axes[2].set_ylabel("top-10 weight (%)"); axes[2].set_xlabel("date")
axes[2].set_title("Concentration — sum of the ten largest weights")

for a in axes: a.grid(alpha=0.25)
plt.tight_layout(); savefig("fig_weight_diagnostics")
plt.show()

# A year of trading is the SUM of its daily turnovers, so the annual figure
# is 252 x the mean. The distribution is right-skewed, so annualising the
# median instead understates it by about a quarter.
print(f"median daily turnover  : {diag_post['turnover'].median():.4f}")
print(f"mean daily turnover    : {diag_post['turnover'].mean():.4f}")
print(f"annualised (252 x mean): {diag_post['turnover'].mean() * CFG.trading_days:.2f}x")
print(f"median gross short     : {diag_post['short_gross'].median():.2%}")
print(f"median top-10 weight   : {diag_post['top10_weight'].median():.2%}")
print("\nRead the turnover figure before the return charts: if rebalancing costs")
print("would swamp the drift differences, the comparison is descriptive about")
print("weighting schemes, not a claim about an investable index.")

## Cell 13 — Cumulative returns

The headline chart, and the reason the notebook exists. All three series are $1
invested in the same universe. There is no scaling constant, no volatility
matching, and no full-sample quantity anywhere in their construction — so the
vertical distances mean what they appear to mean (D2, B3).

In [ ]:
# ═══ CELL 13 ═══
cum = log_df.cumsum()

styles = {"SPY": ("steelblue", 1.3), "RSP": ("darkorange", 1.1),
          "EWMA_PCA": ("seagreen", 1.3)}
labels = {"SPY": "SPY (cap-weighted)", "RSP": "RSP (equal-weighted)",
          "EWMA_PCA": "PC1 (eigen-weighted)"}

fig, ax = plt.subplots(figsize=(13, 6))
for c, (col, lw) in styles.items():
    ax.plot(cum.index, cum[c], color=col, lw=lw, label=labels[c])

ax.axhline(0, ls="--", c="gray", alpha=0.5)
ax.set_title("Cumulative log return — three weighting schemes, same universe, $1 invested")
ax.set_ylabel("cumulative log return"); ax.set_xlabel("date")
ax.legend(loc="upper left"); ax.grid(alpha=0.25)
plt.tight_layout(); savefig("fig_cumulative_returns")
plt.show()

summary = pd.DataFrame({
    "total log return": cum.iloc[-1],
    "ann. drift":       log_df.mean() * CFG.trading_days,
    "ann. vol":         ret_df.std() * np.sqrt(CFG.trading_days),
})
summary["drift / vol (no r_f)"] = summary["ann. drift"] / summary["ann. vol"]

# formatted without.style: that accessor needs jinja2, which is not everywhere
shown = pd.DataFrame({
    "total log return":     summary["total log return"].map("{:.3f}".format),
    "ann. drift":           summary["ann. drift"].map("{:.2%}".format),
    "ann. vol":             summary["ann. vol"].map("{:.2%}".format),
    "drift / vol (no r_f)": summary["drift / vol (no r_f)"].map("{:.2f}".format),
})
display(shown)
print("\nNo risk-free rate is subtracted: these are total returns, and the")
print("last column is not a Sharpe ratio in the published sense.")

## Cell 14 — Rolling one-year drift

In [ ]:
# ═══ CELL 14 ═══
rolling = log_df.rolling(CFG.trading_days).mean() * CFG.trading_days

fig, ax = plt.subplots(figsize=(13, 6))
for c, (col, lw) in styles.items():
    ax.plot(rolling.index, rolling[c], color=col, lw=lw, label=labels[c])
ax.axhline(0, ls="--", c="gray", alpha=0.6)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_title("Rolling 252-day annualised drift")
ax.set_ylabel("annualised drift"); ax.set_xlabel("date")
ax.legend(loc="lower left"); ax.grid(alpha=0.25)
plt.tight_layout(); savefig("fig_rolling_drift")
plt.show()

print("Overlapping windows: roughly "
      f"{len(rolling.dropna()) / CFG.trading_days:.0f} independent observations "
      "underlie this chart, not thousands.")

## Cell 15 — Divergence across horizons

Measured against **both** benchmarks, since PC1 differs from cap-weighting and
from equal-weighting for different reasons. Each panel scales to its own
horizon — `sharey` is deliberately off, and no `set_ylim` is shared.

In [ ]:
# ═══ CELL 15 ═══
d_spy = log_df["EWMA_PCA"] - log_df["SPY"]
d_rsp = log_df["EWMA_PCA"] - log_df["RSP"]

windows = [(252, "252d (1Y)"), (126, "126d (6M)"), (63, "63d (3M)"), (21, "21d (1M)")]
fig, axes = plt.subplots(len(windows) + 1, 1, figsize=(13, 16), sharex=True)

axes[0].plot(cum.index, cum["EWMA_PCA"] - cum["SPY"], lw=0.9,
             color="steelblue", label="PC1 − SPY")
axes[0].plot(cum.index, cum["EWMA_PCA"] - cum["RSP"], lw=0.9,
             color="darkorange", label="PC1 − RSP")
axes[0].axhline(0, ls="--", c="gray", alpha=0.5)
axes[0].set_ylabel("cumulative gap"); axes[0].legend(loc="upper left")
axes[0].set_title("Cumulative divergence since start of window")

for ax, (w, lbl) in zip(axes[1:], windows):
    ax.plot(d_spy.index, d_spy.rolling(w).sum(), lw=0.8, color="steelblue", label="vs SPY")
    ax.plot(d_rsp.index, d_rsp.rolling(w).sum(), lw=0.8, color="darkorange",
            alpha=0.85, label="vs RSP")
    ax.axhline(0, ls="--", c="gray", alpha=0.5)
    ax.set_ylabel("divergence"); ax.set_title(f"Rolling {lbl} cumulative divergence")
    ax.legend(loc="upper left", fontsize=8)

axes[-1].set_xlabel("date")
for a in axes: a.grid(alpha=0.25)
plt.tight_layout(); savefig("fig_divergence")
plt.show()

## Cell 16 — Annual drift, with confidence intervals

Point estimates alone cannot carry a comparison between years. At the roughly
1.2% daily volatility of these series a one-year mean drift carries an
annualised standard error near 19 percentage points, so year-on-year gaps of
the size seen here sit well inside the noise.

The **stationary bootstrap** (Politis–Romano) resamples blocks whose lengths are
Geometric with mean `boot_block`, which preserves volatility clustering and
autocorrelation. Fixed-length blocks would leave the resampled series
non-stationary; an i.i.d. bootstrap would destroy both and understate the
interval.

In [ ]:
# ═══ CELL 16 ═══
def stationary_bootstrap_ci(x, stat, n_boot, mean_block, seed, alpha=0.05):
    rng = np.random.default_rng(seed)
    x = np.asarray(pd.Series(x).dropna(), dtype=float)
    n = len(x)
    if n < mean_block * 2:
        return np.nan, np.nan

    p = 1.0 / mean_block
    out = np.empty(n_boot)
    for b in range(n_boot):
        lens, starts, total = [], [], 0
        while total < n:
            L = int(rng.geometric(p))
            lens.append(L)
            starts.append(int(rng.integers(0, n)))
            total += L
        offs = np.concatenate([np.arange(L) for L in lens])[:n]
        base = np.repeat(np.array(starts), np.array(lens))[:n]
        out[b] = stat(x[(base + offs) % n])
    return tuple(np.quantile(out, [alpha / 2, 1 - alpha / 2]))


def ann(a):
    return a.mean() * CFG.trading_days


rows = []
for year, block in log_df.groupby(log_df.index.year):
    if len(block) < 200:
        continue                                   # incomplete year
    row = {"Year": year, "days": len(block)}
    for c in ["SPY", "RSP", "EWMA_PCA"]:
        pt = ann(block[c].to_numpy())
        lo, hi = stationary_bootstrap_ci(block[c], ann, CFG.boot_n,
                                         CFG.boot_block, CFG.seed)
        row[c] = f"{pt:+.1%}  [{lo:+.0%}, {hi:+.0%}]"
    rows.append(row)

annual = pd.DataFrame(rows).set_index("Year")
print("Annualised drift, 95% stationary-block-bootstrap CI in brackets\n")
display(annual)

print("\nPairwise differences over the full window (PC1 minus benchmark):")
for name, series in [("SPY", d_spy), ("RSP", d_rsp)]:
    pt = ann(series.to_numpy())
    lo, hi = stationary_bootstrap_ci(series, ann, CFG.boot_n, CFG.boot_block, CFG.seed)
    sig = "significant" if (lo > 0) == (hi > 0) else "NOT significant"
    print(f"  vs {name}: {pt:+.2%} per year   95% CI [{lo:+.2%}, {hi:+.2%}]   {sig}")

print("\nNote: the rolling chart in Cell 14 and this table are computed on the same")
print("window; incomplete years are excluded from BOTH, so they cannot disagree.")

## Cell 17 — Spectrum and loadings on the final day

The old notebook printed 99% positive loadings inside the loop and 0% here,
because this cell re-ran the eigendecomposition without re-applying the sign
flip, and indexed a reversed eigenvalue array against unreversed
eigenvectors.

Both are fixed structurally rather than by repeating the rule: Cell 8 exports
the exact matrix it decomposed on the final usable day, together with the active
index and the aligned eigenvector it produced. This cell re-derives from those,
so agreement is a genuine check on the export path and is asserted, not hoped
for.

Note it must be `S_{t-1}` — the state the loop *used* — not the post-update
`S_final`, which the loop never saw.

In [ ]:
# ═══ CELL 17 ═══
S_used, idx_last, v_loop, sg_last = LAST

evals, evecs = np.linalg.eigh(S_used)
order = np.argsort(evals)[::-1]
evals, evecs = evals[order], evecs[:, order]      # reverse BOTH

pc1 = evecs[:, 0]
if np.dot(pc1, v_loop) < 0:                       # align to the loop's own vector
    pc1 = -pc1

# Rebuild the weights exactly as Cell 8 did, so this cell checks the whole
# path -- eigenvector, sigma, normalisation -- and not just the eigenvector.
raw_last = pc1 / sg_last
w_last = raw_last / raw_last.sum()

# Orient the plotted loadings the way the weights point. The eigenvector's
# sign carries no information (v and -v are the same mode) and w is invariant
# to it, so the readable representative is the one whose net exposure is long.
pc1_oriented = pc1 if raw_last.sum() > 0 else -pc1

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].bar(range(1, 21), evals[:20] / evals.sum() * 100, color="steelblue")
axes[0].set_xlabel("principal component"); axes[0].set_ylabel("variance explained (%)")
axes[0].set_title("Scree plot — EWMA covariance, final day")

axes[1].hist(pc1_oriented, bins=50, color="seagreen", edgecolor="white")
axes[1].axvline(0, ls="--", c="gray")
axes[1].set_xlabel("loading (oriented long)"); axes[1].set_ylabel("stocks")
axes[1].set_title("PC1 loading distribution")

for a in axes: a.grid(alpha=0.25)
plt.tight_layout(); savefig("fig_scree_loadings")
plt.show()

ve_here, pp_here = evals[0] / evals.sum(), (w_last > 0).mean() * 100
last_day = diag_post["pc1"].last_valid_index()
ve_loop = diag_post.loc[last_day, "var_explained"]
pp_loop = diag_post.loc[last_day, "pct_positive"]

print(f"this cell : variance explained {ve_here:.2%}   percent positive {pp_here:.2f}%")
print(f"the loop  : variance explained {ve_loop:.2%}   percent positive {pp_loop:.2f}%")
assert round(ve_here * 100, 2) == round(ve_loop * 100, 2), "variance explained disagrees"
assert round(pp_here, 2) == round(pp_loop, 2), "percent positive disagrees"
print("\nOK: Cell 17 reproduces Cell 8 to 2dp.")

## Cell 18 — 2020 in detail

The old notebook annotated this period with a mechanism — EWMA memory carrying
pre-crisis loadings into the shock — that was asserted rather than measured
, on a series whose level was a scaling artefact anyway. Here the
diagnostics are plotted alongside the returns and no mechanism is claimed: what
the weights actually did is on the page, and the reading is left to you.

In [ ]:
# ═══ CELL 18 ═══
z0, z1 = "2019-06-01", "2021-06-30"
sl = ret_df.loc[z0:z1]
cz = np.log1p(sl).cumsum()
dz = diag_post.loc[z0:z1]

fig, axes = plt.subplots(3, 1, figsize=(13, 11), sharex=True)

for c, (col, lw) in styles.items():
    axes[0].plot(cz.index, cz[c], color=col, lw=lw, label=labels[c])
axes[0].set_ylabel("cumulative log return"); axes[0].legend(loc="upper left", fontsize=8)
axes[0].set_title("Cumulative returns — COVID window (rebased to window start)")

axes[1].plot(dz.index, dz["var_explained"] * 100, lw=0.9, color="darkorange")
ax1b = axes[1].twinx()
ax1b.plot(dz.index, dz["pct_positive"], lw=0.9, color="steelblue")
axes[1].set_ylabel("variance explained (%)", color="darkorange")
ax1b.set_ylabel("% positive weights", color="steelblue")
axes[1].set_title("Did the factor structure change, and when?")

axes[2].plot(dz.index, dz["turnover"], lw=0.8, color="indianred")
axes[2].set_yscale("log"); axes[2].set_ylabel("daily turnover")
axes[2].set_xlabel("date"); axes[2].set_title("PC1 rebalancing intensity")

for a in axes:
    a.axvline(pd.Timestamp("2020-02-19"), ls=":", c="gray", alpha=0.8)
    a.axvline(pd.Timestamp("2020-03-23"), ls=":", c="red", alpha=0.8)
    a.grid(alpha=0.25)
plt.tight_layout(); savefig("fig_covid_zoom")
plt.show()

crisis  = np.log1p(ret_df.loc["2020-02-19":"2020-03-23"]).sum()
recover = np.log1p(ret_df.loc["2020-03-24":"2020-12-31"]).sum()
print("cumulative log return, peak to trough (19 Feb - 23 Mar 2020):")
print(crisis.map("{:+.2%}".format).to_string())
print("\ncumulative log return, trough to year end:")
print(recover.map("{:+.2%}".format).to_string())

## Cell 19 — Acceptance tests

All of section 6, not just future invariance. The suite asserts at the end: if
anything fails the notebook stops, because a pipeline that fails these is not
reporting what it claims to report.

The headline test re-runs the pipeline with all data after a cutoff replaced by
noise and requires every earlier output to be **bit-identical**. That one test
catches B3, B6, B7 and B9 simultaneously.

In [ ]:
# ═══ CELL 19 ═══
RESULTS = []


def check(name, ok, detail=""):
    RESULTS.append((name, bool(ok), detail))
    print(f"  {'PASS' if ok else 'FAIL'}  {name}" + (f"   {detail}" if detail else ""))


# --- 1. future invariance -------------------------------------------------
print("[1] Future invariance")
CUT = len(DATES) // 2
rng = np.random.default_rng(CFG.seed)
rets_p = rets.copy()
noise = rng.normal(0, 0.35, (len(DATES) - CUT, rets.shape[1]))
rets_p.iloc[CUT:] = np.where(valid.to_numpy()[CUT:], noise, np.nan)

vol_p = ewma_vol_lagged(rets_p, valid, CFG.vol_half_life,
                        CFG.vol_floor, CFG.vol_warmup_obs)
diag_p, _, _ = run_streaming_pca(rets_p, valid, vol_p, MEMBERS, CFG.pca_half_life,
                                 CFG.min_names, CFG.z_clip, CFG.min_net_gross,
                                 CFG.rot_lags)

ok_fi = np.allclose(vol.iloc[:CUT].to_numpy(), vol_p.iloc[:CUT].to_numpy(),
                    equal_nan=True, rtol=0, atol=0)
inv_cols = ["pc1", "pct_positive", "var_explained", "turnover",
            "short_gross", "top10_weight",
            "lam1_norm", "mp_edge", "n_above_mp", "share_above_mp"] \
           + [f"rot_{h}d" for h in CFG.rot_lags]
for col in inv_cols:
    ok_fi &= np.allclose(diag[col].iloc[:CUT].to_numpy(), diag_p[col].iloc[:CUT].to_numpy(),
                         equal_nan=True, rtol=0, atol=0)
check("every pre-cutoff output bit-identical", ok_fi,
      f"{len(inv_cols)} diagnostics + vol, cutoff {DATES[CUT]:%Y-%m-%d}, rtol=0 atol=0")

# The comparison above stops one row short of the perturbation, so it cannot
# see one-day look-ahead: an estimator that standardised r_t with sigma_t
# would read only unperturbed data on every row it checks. The cutoff row is
# where the noise lands, so everything fixed before r_t is read must be
# identical there too. pc1 is excluded: it reads r_t by construction.
_lag_cols = [c for c in inv_cols if c != "pc1"]
ok_lag = np.allclose(vol.iloc[CUT].to_numpy(), vol_p.iloc[CUT].to_numpy(),
                     equal_nan=True, rtol=0, atol=0)
for col in _lag_cols:
    ok_lag &= np.allclose(diag[col].iloc[CUT], diag_p[col].iloc[CUT],
                          equal_nan=True, rtol=0, atol=0)
check("cutoff-row outputs unchanged, so no one-day look-ahead", ok_lag,
      f"{len(_lag_cols)} diagnostics + vol on {DATES[CUT]:%Y-%m-%d}, pc1 excluded")

# and the other side of it: if the perturbation did not move pc1 on that row,
# the noise never reached the quantity under test and the check above is empty
_a, _b = diag["pc1"].iloc[CUT], diag_p["pc1"].iloc[CUT]
check("the perturbation does reach the cutoff row",
      (not np.isfinite(_a)) or (not np.isfinite(_b)) or _a != _b,
      f"pc1 {_a:.6g} -> {_b:.6g}")

# --- 2. NaN containment ---------------------------------------------------
print("\n[2] NaN containment")
probe_col = vol.notna().sum().idxmax()
r2, v2 = rets[[probe_col]].copy(), valid[[probe_col]].copy()
hit = int(np.flatnonzero(v2.to_numpy().ravel())[len(DATES) // 2])
r2.iloc[hit, 0], v2.iloc[hit, 0] = np.nan, False
vol2 = ewma_vol_lagged(r2, v2, CFG.vol_half_life, CFG.vol_floor, CFG.vol_warmup_obs)
after = vol2.iloc[hit + 1:, 0]
check("one missing return -> zero trailing NaN variances",
      int(after.isna().sum()) == 0, f"{probe_col}: trailing NaNs = {int(after.isna().sum())}")

# --- 3. no standardisation cap -------------------------------------------
print("\n[3] No standardisation cap")
check("lagged standardisation reports moves beyond the cap",
      np.abs(z_all).max() > cap, f"max|z| = {np.abs(z_all).max():.2f} vs cap {cap:.2f}")
_v = np.full(rets.shape[1], np.nan); _mx = 0.0
_R, _M = rets.to_numpy(), valid.to_numpy()
for t in range(len(DATES)):                       # the old contemporaneous form
    x = np.where(_M[t], _R[t], 0.0) ** 2
    _v = np.where(_M[t], np.where(np.isfinite(_v), LAMBDA_VOL * _v + (1 - LAMBDA_VOL) * x, x), _v)
    zz = np.abs(np.where(_M[t], _R[t], np.nan) / np.sqrt(_v))
    if np.isfinite(zz).any():
        _mx = max(_mx, np.nanmax(zz))
check("contemporaneous standardisation is structurally capped", _mx <= cap + 1e-9,
      f"max|z| contemporaneous = {_mx:.2f} <= {cap:.2f}")

# --- 4. lag verified both ways -------------------------------------------
print("\n[4] Lag verified both ways")
r4 = rets[[probe_col]].copy()
v4 = valid[[probe_col]].copy()
t0 = hit
r4.iloc[t0, 0] = r4.iloc[t0, 0] + 0.20
vol4 = ewma_vol_lagged(r4, v4, CFG.vol_half_life, CFG.vol_floor, CFG.vol_warmup_obs)
base = ewma_vol_lagged(rets[[probe_col]], valid[[probe_col]], CFG.vol_half_life,
                       CFG.vol_floor, CFG.vol_warmup_obs)
check("perturbing r_t leaves sigma_t unchanged",
      np.allclose(base.iloc[t0, 0], vol4.iloc[t0, 0], equal_nan=True, rtol=0, atol=0))
check("perturbing r_t changes sigma_{t+1}",
      not np.allclose(base.iloc[t0 + 1, 0], vol4.iloc[t0 + 1, 0], equal_nan=True))

# --- 5. Cell 17 agrees with Cell 8 ---------------------------------------
print("\n[5] Cell 17 reproduces Cell 8")
check("variance explained agrees to 2dp",
      round(ve_here * 100, 2) == round(ve_loop * 100, 2),
      f"{ve_here*100:.2f}% vs {ve_loop*100:.2f}%")
check("percent positive agrees to 2dp", round(pp_here, 2) == round(pp_loop, 2),
      f"{pp_here:.2f}% vs {pp_loop:.2f}%")

# --- 6. membership semantics ---------------------------------------------
print("\n[6] Membership semantics")

# --- no look-ahead from the change log itself -------------------------------
# The future-invariance test above perturbs returns. Membership is a separate
# input: the timeline is rewound from TODAY's constituent list, so a change
# dated after the window must be undone rather than skipped, or a later
# addition is carried back across the whole sample and a later removal is
# dropped from it.
_late = changes_df[changes_df["date"] > AS_OF]
_late_add = sorted({t for t in _late["added"].dropna() if str(t) != "<NA>"})
_late_rem = sorted({t for t in _late["removed"].dropna() if str(t) != "<NA>"})

if not len(_late):
    print("  SKIP  no index changes dated after the window end")
else:
    _probe = pd.date_range(CFG.start_date, CFG.end_date, freq="7D")
    _names = sorted(set(_late_add) | set(_late_rem))
    _M = pd.DataFrame(membership_matrix(TIMELINE, _probe, _names),
                      index=_probe, columns=_names)

    # a name added after the window must never appear inside it, unless its own
    # record shows an earlier spell of membership
    _early = {}
    for t in _names:
        prior = changes_df[(changes_df["date"] <= AS_OF)
                           & ((changes_df["added"] == t)
                              | (changes_df["removed"] == t))]
        _early[t] = len(prior) > 0

    _bad_in = [t for t in _late_add if not _early[t] and bool(_M[t].any())]
    check("a name added after the window is not a member inside it",
          not _bad_in,
          f"{len(_late_add)} late additions, leaked: {_bad_in}")

    # a name removed after the window was a member until that removal, so it
    # must be present at the window end
    _bad_out = [t for t in _late_rem if not bool(_M[t].iloc[-1])]
    check("a name removed after the window is still a member at its end",
          not _bad_out,
          f"{len(_late_rem)} late removals, dropped: {_bad_out}")

_probe6 = pd.to_datetime(["2016-06-01", "2019-06-01", "2020-06-01", "2023-06-01"])
_mm = membership_matrix(TIMELINE, _probe6, UNIVERSE)
if "EQT" in _mm.columns:
    check("re-entrant absent during its index gap",
          bool(_mm.loc[_probe6[0], "EQT"]) and not bool(_mm.loc[_probe6[1], "EQT"])
          and not bool(_mm.loc[_probe6[2], "EQT"]) and bool(_mm.loc[_probe6[3], "EQT"]),
          "EQT: in 2016, out 2019/2020, back 2023")
    check("re-entrant kept in the analysis universe (traded through the gap)",
          "EQT" in KEEP)
if "FOXA" in _mm.columns:
    check("same-day rename stays present in the earlier period",
          bool(_mm.loc[_probe6[0], "FOXA"]), "FOXA present in 2016")

# --- 7. gap rule ----------------------------------------------------------
print("\n[7] Gap rule")
_idx = pd.to_datetime(["2020-01-02", "2020-01-03", "2020-01-20", "2020-01-21"])
_px = pd.DataFrame({"X": [100.0, 101.0, 130.0, 131.0]}, index=_idx)
_r, _v = simple_returns(_px, CFG.max_gap_days)
check("multi-day gap invalidates the return, not a giant 1-day move",
      (not bool(_v.iloc[2, 0])) and bool(_v.iloc[1, 0]) and np.isnan(_r.iloc[2, 0]))
_neg = pd.DataFrame({"X": [10.0, -1.0, 5.0]}, index=pd.bdate_range("2020-01-01", periods=3))
check("non-positive price guarded", not bool(simple_returns(_neg, CFG.max_gap_days)[1].iloc[1, 0]))

# --- 8. recovery on synthetic data ---------------------------------------
print("\n[8] Recovery on synthetic data")
_rng = np.random.default_rng(CFG.seed)
_T, _N = 1500, 80
_d = pd.bdate_range(CFG.start_date, periods=_T)
_lv = np.zeros(_T)
for t in range(1, _T):
    _lv[t] = 0.97 * _lv[t - 1] + _rng.normal(0, 0.15)
_m = _rng.normal(0, 1, _T) * 0.01 * np.exp(_lv - _lv.var() / 2)
_b = np.clip(_rng.normal(1.0, 0.25, _N), 0.3, None)
_r = _m[:, None] * _b[None, :] + _rng.normal(0, 1, (_T, _N)) * _rng.uniform(.008,.02, _N)
_cols = [f"S{i:03d}" for i in range(_N)]
_rets = pd.DataFrame(_r, index=_d, columns=_cols)
_valid = pd.DataFrame(True, index=_d, columns=_cols)
_mem = pd.DataFrame(True, index=_d, columns=_cols)
_vol = ewma_vol_lagged(_rets, _valid, CFG.vol_half_life, CFG.vol_floor, CFG.vol_warmup_obs)
_diag, _, _ = run_streaming_pca(_rets, _valid, _vol, _mem, CFG.pca_half_life,
                                CFG.min_names, CFG.z_clip, CFG.min_net_gross,
                                CFG.rot_lags)
_j = pd.concat([_diag["pc1"], pd.Series(_m, index=_d, name="truth")],
               axis=1).iloc[CFG.burn_in_days:].dropna()
_corr = _j.iloc[:, 0].corr(_j.iloc[:, 1])
_ratio = _j.iloc[:, 0].std() / _j.iloc[:, 1].std()
check("PC1 correlates > 0.95 with the planted factor", _corr > 0.95, f"corr = {_corr:.4f}")
check("PC1 vol within ~2x of the true market vol", 0.5 < _ratio < 2.0, f"ratio = {_ratio:.3f}")

# --- summary --------------------------------------------------------------
_fail = [n for n, ok, _ in RESULTS if not ok]
print("\n" + "=" * 68)
print(f"{len(RESULTS) - len(_fail)}/{len(RESULTS)} passed")
for n in _fail:
    print(f"  FAILED: {n}")
print("=" * 68)
assert not _fail, f"acceptance tests failed: {_fail}"
print("PASS - no look-ahead anywhere in the pipeline; all section 6 tests green.")

## Cell 20 — Loading trajectory

The literature treats drifting factor loadings as something an estimator must
survive, not as something to read. O-GARCH fixes the rotation after one
decomposition; GO-GARCH replaces it with a static map; consistency results treat
drift as tolerable noise. In each case the path of the loadings is a nuisance.

Here it is the object of interest, so it is measured directly. For lags `h` the
cell reports the principal angle between `v_t` and `v_{t-h}`, in degrees,

$$\theta_t(h) = \angle\big(v_t,\, v_{t-h}\big),$$

computed on the names active on **both** dates and renormalised, so entries and
exits do not register as rotation. The angle is taken between the lines the two
vectors span, since the sign of an eigenvector is arbitrary.

Turnover (Cell 12) is not a substitute: it measures movement in `w = v/sigma`,
which also moves when volatility moves, and is denominated in portfolio weight
rather than in angle.

**Sensitivity depends on `h` relative to the covariance half-life.** On
synthetic panels with a planted change in loadings (`check_planted.py`), the
median 21-day angle rose by a factor of 1.5 against an unchanged panel while
the median 252-day angle rose by 3.3. The long-horizon response is also
delayed: the 252-day ratio is 2.4 over the year following the change and 4.2
over the year after that. At a 252-day half-life the estimator needs roughly a
year to absorb a change in structure, so a break appears as sustained rotation
at the longer horizon rather than a spike at the shorter one. Read `rot_252d`
as the structural signal and `rot_21d` as the noise floor.

In [ ]:
# ═══ CELL 20 ═══
rot_cols = [f"rot_{h}d" for h in CFG.rot_lags]
rot_post = diag_post[rot_cols]

fig, ax = plt.subplots(figsize=(13, 5))
for h, c in zip(CFG.rot_lags, ["steelblue", "darkorange", "seagreen"]):
    ax.plot(rot_post.index, rot_post[f"rot_{h}d"], lw=0.8, label=f"{h}-day lag")
ax.set_ylabel("principal angle (degrees)"); ax.set_xlabel("date")
ax.set_title("PC1 loading rotation: angle between $v_t$ and $v_{t-h}$")
ax.legend(loc="upper right"); ax.grid(alpha=0.25)
plt.tight_layout(); savefig("fig_rotation")
plt.show()

summary_rot = pd.DataFrame({
    "median": rot_post.median(),
    "p90":    rot_post.quantile(0.90),
    "max":    rot_post.max(),
    "annualised (deg/yr)": rot_post.median() * (CFG.trading_days / np.array(CFG.rot_lags)),
})
print("Rotation of the leading eigenvector, degrees")
display(summary_rot.round(2))

# Did the rotation respond when market structure broke?
# 2020 is split rather than taken whole: the crash and the months after it
# move in opposite directions, so a calendar-year median averages them away.
# The crash window runs from the pre-crash high to the end of April.
win = {"2018 (volatility spikes)":       ("2018-01-01", "2018-12-31"),
       "2020 crash (19 Feb - 30 Apr)":   ("2020-02-19", "2020-04-30"),
       "2020 after the crash (May-Dec)": ("2020-05-01", "2020-12-31"),
       "2022 (rate tightening)":         ("2022-01-01", "2022-12-31"),
       "remainder of sample":            (None, None)}
rows = []
mask_all = pd.Series(True, index=rot_post.index)
for label, (a, b) in win.items():
    m = mask_all.copy() if a is None else ((rot_post.index >= a) & (rot_post.index <= b))
    if a is None:
        for a2, b2 in [v for k, v in win.items() if v[0] is not None]:
            m &= ~((rot_post.index >= a2) & (rot_post.index <= b2))
    rows.append(dict(period=label, days=int(np.sum(m)),
                     **{f"rot_{h}d": float(np.nanmedian(rot_post[f"rot_{h}d"][m]))
                        for h in CFG.rot_lags}))
print("")
print("Median rotation by period (degrees)")
display(pd.DataFrame(rows).set_index("period").round(2))

# The calendar-year summary of 2020 against the month that drove it, which is
# the comparison Section 4.2.4 rests on, and the date of each maximum.
_y20 = rot_post.loc["2020"]
_m20 = rot_post.loc["2020-03"]
print("")
print("2020 taken whole, against March alone (median degrees)")
for h in CFG.rot_lags:
    print(f"   rot_{h}d: calendar 2020 {np.nanmedian(_y20[f'rot_{h}d']):6.2f}   "
          f"March 2020 {np.nanmedian(_m20[f'rot_{h}d']):6.2f}")
print("")
print("maximum of each horizon")
for h in CFG.rot_lags:
    _c = rot_post[f"rot_{h}d"]
    print(f"   rot_{h}d: {_c.max():6.2f} on {_c.idxmax():%Y-%m-%d}")

## Cell 21 — Spectral separation

Whether a dynamically updated eigenvector should be read as structure or shrunk
away as noise is an empirical question about the spectrum, so this cell measures
it rather than assuming either side.

For each day the eigenvalues of the active block of `S_{t-1}` are rescaled to
sum to the number of active names, as a correlation matrix's would, and compared
with the Marčenko-Pastur upper edge for pure noise,

$$\lambda_+ = \Big(1 + \sqrt{N_t / T_{\text{eff}}}\Big)^2,
\qquad T_{\text{eff}} = \frac{1+\lambda_S}{1-\lambda_S} \approx 727.$$

Three quantities are reported: the leading eigenvalue, the number of eigenvalues
above the edge, and the share of total variance they carry. A leading eigenvalue
far above `lambda_+` means PC1 is a spike that noise cannot produce, and the
case for shrinking it away is weak. A leading eigenvalue near the edge would
mean the opposite. The count above the edge also indicates how many components
are separated from the bulk at all, which bounds how much of the spectrum can be
read as structure.

Note this does not rescue the *variance share* of PC1, which stays biased upward
at `N/T_eff ~ 0.65`. Separation and unbiasedness are different claims.

In [ ]:
# ═══ CELL 21 ═══
sp = diag_post[["lam1_norm", "mp_edge", "n_above_mp", "share_above_mp", "n_active"]].dropna()

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
axes[0].plot(sp.index, sp["lam1_norm"], lw=0.8, color="steelblue", label=r"$\lambda_1$")
axes[0].plot(sp.index, sp["mp_edge"], lw=1.0, color="crimson", ls="--",
             label=r"MP edge $\lambda_+$")
axes[0].set_yscale("log"); axes[0].set_ylabel("eigenvalue (log scale)")
axes[0].set_title("Leading eigenvalue against the Marčenko–Pastur noise edge")
axes[0].legend(loc="center right"); axes[0].grid(alpha=0.25)

axes[1].plot(sp.index, sp["n_above_mp"], lw=0.8, color="seagreen")
axes[1].set_ylabel("eigenvalues above the edge"); axes[1].set_xlabel("date")
axes[1].set_title("How many components separate from the bulk")
axes[1].grid(alpha=0.25)
plt.tight_layout(); savefig("fig_spectral")
plt.show()

ratio = sp["lam1_norm"] / sp["mp_edge"]
print(f"lambda_1 / MP edge      : median {ratio.median():.1f}x   "
      f"min {ratio.min():.1f}x   max {ratio.max():.1f}x")
print(f"days with lambda_1 > edge : {(ratio > 1).mean():.2%}")
print(f"eigenvalues above edge  : median {sp['n_above_mp'].median():.0f} "
      f"of {sp['n_active'].median():.0f} active names")
print(f"variance share above edge: median {sp['share_above_mp'].median():.2%}")
print("")
print("A leading eigenvalue far above the edge on essentially every day means")
print("PC1 is a spike that noise alone cannot produce, so shrinking it away would")
print("discard signal. This speaks to separation only. The variance SHARE of")
print("PC1 carries a sampling bias of order q/(lambda-1), about half a per cent")
print("in relative terms here, against a downward bias from the identity prior")
print("that survives burn-in; neither is large enough to matter to any")
print("comparison made in the paper.")